# QUANTIZATION COMPARISON: Min-Max vs Percentile Range Selection

    This notebook demonstrates how OUTLIERS can destroy quantization quality
    and how PERCENTILE-BASED range selection can solve this problem.
    
    KEY CONCEPT: When you have outliers in your data, using min/max directly
    wastes most of your quantization range on values that rarely occur!

In [1]:
import numpy as np

# Suppress scientific notation for cleaner output
np.set_printoptions(suppress=True)

## CREATE TEST DATA WITH AN OUTLIER

In [2]:
# Generate 10,000 random values between -50 and 150
# This represents typical neural network weights/activations
params = np.random.uniform(low=-50, high=150, size=10000)

# INTRODUCE AN OUTLIER: Set the last value to 1000
# This simulates a rare extreme value that can appear in real data
params[-1] = 1000

# Round to 2 decimal places for readability
params = np.round(params, 2)

print(params)

[ 112.3   -34.    138.95 ...   -6.97  136.08 1000.  ]


    MOST VALUES (9,999 samples):
    [-50 <──────────────────────────────────────────> 150]
                            ↑
                    Normal distribution
    
    OUTLIER (1 sample): 1000
    
    This is the problem we're trying to solve!

## QUANTIZATION FUNCTIONS

In [3]:
def clamp(params_q: np.array, lower_bound: int, upper_bound: int) -> np.array:
    """
    Clamp values to valid quantization range [0, 255] for 8-bit.
    Prevents overflow/underflow.
    """
    params_q[params_q < lower_bound] = lower_bound
    params_q[params_q > upper_bound] = upper_bound
    return params_q


def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float, int]:
    """
    STANDARD ASYMMETRIC QUANTIZATION using MIN-MAX range.
    
    Problem with outliers:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    If we have one outlier at 1000 and most data in [-50, 150]:
    
    Range used: [-50 to 1000] = 1050 units
    Scale = 1050 / 255 = 4.12
    
    This means:
    - Most of our data (in 200-unit range) only uses 200/4.12 ≈ 48 quantization levels
    - We waste 255 - 48 = 207 levels on the rare outlier!
    - Precision for normal values suffers dramatically!
    
    Visual:
    Quantized range: [0 <─normal data─> 48 ... unused ... 255 <- outlier]
                         ↑                    ↑              ↑
                    9,999 values        wasted space    1 value
    """
    alpha = np.max(params)  # Will be 1000 (the outlier!)
    beta = np.min(params)   # Around -50
    scale = (alpha - beta) / (2**bits - 1)
    zero = -1 * np.round(beta / scale)
    lower_bound, upper_bound = 0, 2**bits - 1
    quantized = clamp(np.round(params / scale + zero), lower_bound, upper_bound).astype(np.int32)
    return quantized, scale, zero


def asymmetric_quantization_percentile(params: np.array, bits: int, percentile: float = 99.99) -> tuple[np.array, float, int]:
    """
    IMPROVED: PERCENTILE-BASED range selection.
    
    Key idea: Instead of using absolute min/max, use percentiles!
    
    With percentile=99.99:
    - alpha = 99.99th percentile (ignores top 0.01% of values)
    - beta = 0.01th percentile (ignores bottom 0.01% of values)
    
    Benefits:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Range used: [-50 to 150] = 200 units (outlier ignored!)
    Scale = 200 / 255 = 0.78
    
    Now:
    - Normal data uses ALL 255 quantization levels ✓
    - Much better precision for the 99.99% of typical values ✓
    - Outliers get clamped to max value (acceptable trade-off) ✓
    
    Visual:
    Quantized range: [0 <──────── ALL normal data ──────────> 255]
                         ↑                                      ↑
                    9,999 values fully utilized          outlier clamped here
    
    Trade-off: Outlier gets clipped to 150 (was 1000)
    - Original outlier: 1000
    - Reconstructed: ~150
    - ERROR for outlier: LARGE
    - But 99.99% of data has MUCH better precision!
    """
    # Find the percentile values (ignore extreme outliers)
    alpha = np.percentile(params, percentile)        # 99.99th percentile
    beta = np.percentile(params, 100 - percentile)   # 0.01th percentile
    
    scale = (alpha - beta) / (2**bits - 1)
    zero = -1 * np.round(beta / scale)
    lower_bound, upper_bound = 0, 2**bits - 1
    
    # Outliers will be clamped to [0, 255] range
    quantized = clamp(np.round(params / scale + zero), lower_bound, upper_bound).astype(np.int32)
    return quantized, scale, zero


def asymmetric_dequantize(params_q: np.array, scale: float, zero: int) -> np.array:
    """Convert quantized integers back to floating point."""
    return (params_q - zero) * scale


def quantization_error(params: np.array, params_q: np.array):
    """Calculate Mean Squared Error between original and reconstructed."""
    return np.mean((params - params_q)**2)

## QUANTIZE USING BOTH METHODS

In [5]:
# Method 1: Standard min-max (affected by outlier)
(asymmetric_q, asymmetric_scale, asymmetric_zero) = asymmetric_quantization(params, 8)

# Method 2: Percentile-based (robust to outlier)
(asymmetric_q_percentile, asymmetric_scale_percentile, asymmetric_zero_percentile) = \
    asymmetric_quantization_percentile(params, 8)

print(f'Original:')
print(np.round(params, 2))
print()
print(f'Asymmetric (min-max) scale: {asymmetric_scale}, zero: {asymmetric_zero}')
print(asymmetric_q)
print()
print(f'Asymmetric (percentile) scale: {asymmetric_scale_percentile}, zero: {asymmetric_zero_percentile}')
print(asymmetric_q_percentile)

Original:
[ 112.3   -34.    138.95 ...   -6.97  136.08 1000.  ]

Asymmetric (min-max) scale: 4.117647058823529, zero: 12.0
[ 39   4  46 ...  10  45 255]

Asymmetric (percentile) scale: 0.7842157294094071, zero: 64.0
[207  21 241 ...  55 238 255]


    Min-max scale: ~4.12
      Each quantized step = 4.12 units in original scale
      COARSE GRANULARITY (poor precision for normal values)
    
    Percentile scale: ~0.78
      Each quantized step = 0.78 units in original scale  
      FINE GRANULARITY (excellent precision for normal values)
    
    Percentile method has ~5.3x better resolution for typical data!

## DEQUANTIZE BACK TO FLOATS

In [6]:
params_deq_asymmetric = asymmetric_dequantize(asymmetric_q, asymmetric_scale, asymmetric_zero)
params_deq_asymmetric_percentile = asymmetric_dequantize(
    asymmetric_q_percentile, 
    asymmetric_scale_percentile, 
    asymmetric_zero_percentile
)

print(f'Original:')
print(np.round(params, 2))
print('')
print(f'Dequantized (min-max):')
print(np.round(params_deq_asymmetric, 2))
print('')
print(f'Dequantized (percentile):')
print(np.round(params_deq_asymmetric_percentile, 2))

Original:
[ 112.3   -34.    138.95 ...   -6.97  136.08 1000.  ]

Dequantized (min-max):
[ 111.18  -32.94  140.   ...   -8.24  135.88 1000.59]

Dequantized (percentile):
[112.14 -33.72 138.81 ...  -7.06 136.45 149.79]


    Original outlier: 1000.0
    
    Min-max reconstruction: 1000.56
      → Outlier preserved, but at huge cost to other values
    
    Percentile reconstruction: ~149.83
      → Outlier clipped, but 99.99% of data has better precision!
    
    This is the KEY TRADE-OFF of percentile-based quantization.

## EVALUATE QUANTIZATION ERROR (EXCLUDING OUTLIER)

In [7]:
# Calculate MSE for all values EXCEPT the outlier
# This shows the quality for the 99.99% of typical values
print(f'{"Error (min-max) excluding outlier: ":>40}{np.round(quantization_error(params[:-1], params_deq_asymmetric[:-1]), 2)}')
print(f'{"Error (percentile) excluding outlier: ":>40}{np.round(quantization_error(params[:-1], params_deq_asymmetric_percentile[:-1]), 2)}')

     Error (min-max) excluding outlier: 1.39
  Error (percentile) excluding outlier: 0.05


    Error (min-max): ~1.39
    Error (percentile): ~0.05
    
    Percentile method is ~28x better for normal values!

    Use MIN-MAX when:
    ✓ No outliers in your data
    ✓ You need to preserve EVERY value exactly (even outliers)
    ✓ Data distribution is uniform
    
    Use PERCENTILE when:
    ✓ Data contains outliers (common in neural networks!)
    ✓ You care more about typical values than rare extremes
    ✓ Willing to sacrifice outlier precision for overall better quality
    ✓ MOST REAL-WORLD SCENARIOS ← This is the usual choice!

    In neural network quantization:
    - A single outlier activation can destroy model accuracy with min-max
    - Percentile-based quantization (99.9-99.99) is industry standard
    - Modern frameworks (PyTorch, TensorFlow) default to percentile methods
    - Typical choice: 99.9th percentile or 99.99th percentile
    
    THE LESSON: Outliers are the enemy of quantization!
    Smart range selection is crucial for real-world success.